# Topic 4: Population density, urbanisation and projected growth in Cambodia
This notebook reproduces every number on the portal's findings page.
Run **Kernel → Restart & Run All** before submitting. Data must be downloaded first:
`python scripts/download_data.py && python scripts/build_processed.py`.

Each section maps to one course week. Cells marked **TODO** are for the team.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.express as px
from src import config, pipeline, forecast, model, charts
np.random.seed(config.SEED)
pd.set_option("display.max_columns", 50)

## Week 1: the preprocessing pipeline
One documented path from each raw source to the analysis tables.

In [ ]:
report = pipeline.build_processed()
report

In [ ]:
prov = pipeline.load_census()
nat = pipeline.national_series()
com = pipeline.load_communes()
prov.head()

**Data dictionary.** TODO: complete the meaning column for every column used.

In [ ]:
dictionary = pd.DataFrame({
    "column": prov.columns,
    "dtype": prov.dtypes.astype(str).values,
    "example": prov.iloc[0].values,
    "meaning": "",            # TODO
    "source": "S1 census 2019 / derived",
})
dictionary

## Week 2: missing values
A missingness report per source and the reason each gap exists.

In [ ]:
miss = pd.concat([pipeline.missingness_report(d, n) for n, d in
                  {"province": prov, "national": nat, "commune": com}.items() if not d.empty])
miss[miss["missing"] > 0]

In [ ]:
# Missingness map for the national series (years x columns)
m = nat.set_index("year").isna().astype(int)
px.imshow(m.T, aspect="auto", color_continuous_scale=["#E6F1FB", "#0C447C"],
          title="Missing values by year (dark = missing)")

TODO: explain each gap (e.g. World Bank urban share starts in 1960; no urban share is projected after 2024; first-year growth is empty by construction). If you fill anything, show before and after.

## Week 3: outliers
Flagged with fences / robust scores, explained, never deleted.

In [ ]:
prov[["province", "density_calc", "density_calc_score", "density_calc_outlier",
      "household_size", "household_size_score", "household_size_outlier"]].sort_values("density_calc", ascending=False).head(8)

In [ ]:
px.box(prov, y="density_calc", points="all", hover_name="province", log_y=True,
       title="Phnom Penh sits far above the upper fence (log scale)")

TODO: one sentence per flagged province explaining why the value is real or an error.

## Week 4: normalisation
Every map uses per-km² or per-household values. The model uses StandardScaler.

In [ ]:
prov[["province", "population", "density_calc", "households_per_km2", "pop_share_pct"]].describe()

TODO: justify log10 for density (skew) and StandardScaler for the model (logistic regression is scale-sensitive). Show the distribution before and after on the same axes.

## Week 5: categorical encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
if not com.empty:
    oh = OneHotEncoder(handle_unknown="ignore").fit(com[["pro_code"]].astype(str))
    print("One-hot width for province:", len(oh.get_feature_names_out()))
    te = com.groupby("pro_code")["is_sangkat"].mean().rename("target_enc")
    print("Target encoding width: 1 (must be fitted inside CV folds to avoid leakage)")

In [ ]:
pipeline.province_match_table(pd.Series(["Siemreap", "Krong Phnom Penh", "Rotanak Kiri", "Kratié"]))

## Week 6: feature engineering
Each feature written as a claim about the world.

| Feature | Claim |
|---|---|
| `log10_density` | Density matters multiplicatively: going from 10 to 100 per km² is as big a change as 100 to 1,000 |
| `pop_growth_pct` | The pace of growth, not the level, is what changes service needs |
| `pop_growth_roll5` | Single-year moves are noise; the 5-year mean is the trend |
| `urban_pct_change` | Urbanisation speed, in percentage points per year |
| `villages` (commune) | Rural communes are made of many scattered villages |
| TODO | add yours |

In [ ]:
nat[["year", "population", "pop_growth_pct", "pop_growth_roll5", "urban_pct", "urban_pct_change"]].dropna().tail()

## Week 8: feature selection and PCA

In [ ]:
cols = ["log10_density", "household_size", "households_per_km2", "sex_ratio", "pop_share_pct"]
corr = prov[cols].corr(method="spearman")
px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu",
          title="Spearman correlation between province indicators")

In [ ]:
idx, var = model.pca_index(prov, cols)
prov["urbanisation_index"] = idx
print(f"PC1 keeps {var:.0%} of the variance")
prov.sort_values("urbanisation_index", ascending=False)[["province", "urbanisation_index"]].head()

## Week 9: data integration
Grain and key of each source, row counts before/after every join, name-matching table.

In [ ]:
gj, match = pipeline.load_province_geojson()
print("boundary features:", len(match), "| matched:", match["pro_code"].notna().sum())
joined = prov.merge(match, on="pro_code", how="outer", indicator=True)
joined["_merge"].value_counts()

In [ ]:
if not com.empty:
    before = len(com)
    j = com.merge(prov[["pro_code", "province", "density_calc"]], on="pro_code", how="left")
    print(f"communes before join: {before}, after: {len(j)}, without province: {j['province'].isna().sum()}")

## Week 10: imbalanced data: which communes are sangkat?

In [ ]:
if not com.empty:
    res = model.run(com, cost_fn=3, cost_fp=1)
    print(f"Base rate: {res['base_rate']:.1%} of {res['n']} communes")
    print("Threshold chosen from costs (FN=3, FP=1):", res["threshold"])
    print("CV  :", {k: round(v, 3) if isinstance(v, float) else v for k, v in res["cv"].items()})
    print("Test:", {k: round(v, 3) if isinstance(v, float) else v for k, v in res["test"].items()})

In [ ]:
if not com.empty:
    leaky = model.run(com, use_district_type=True)
    print("Recall with district_type:", round(leaky["test"]["recall"], 3),
          "| without:", round(res["test"]["recall"], 3))

TODO: explain why accuracy is the wrong score and why `district_type` is leaky.

## Week 11: time series
Common annual frequency, trend separated, split in time, forecast vs. naive baseline.

In [ ]:
est = nat[nat["kind"] == "estimate"].set_index("year")["population"]
preds, scores = forecast.evaluate(est, test_start=2015)
scores

In [ ]:
charts.population_chart(nat, preds)

TODO: no seasonal component exists in annual data; say so. Does Holt beat the naive baseline?

## Week 12: visualisation
Every chart titled with its finding; class breaks stated.

In [ ]:
fig, r = charts.scatter_density_household(prov)
fig

In [ ]:
charts.choropleth(prov, gj, "density_calc", "People per km², 2019 census")

## Week 13: one pipeline
All fitted steps live in `model.build_pipeline`; CV on training rows; test set used once.

In [ ]:
if not com.empty:
    print(res["pipeline"])

## Week 14: ethics
TODO (write 1–2 paragraphs each):
- **Who is missing:** Cambodians working abroad, people without fixed address, the 2008→2019 boundary changes.
- **Small-area risk:** village-level counts cross-tabulated by sex or age can identify households; we publish only province and commune aggregates.
- **Labels:** "urban" here means the administrative label *sangkat*, not a measured density; a rule based on it may treat rural sangkat unfairly.
- **Licence:** the Gazetteer has no licence; we republish only derived counts.

## Findings (copy to the portal's findings page)

In [ ]:
pp = prov[prov["province"] == "Phnom Penh"].iloc[0]
print(f"1. Phnom Penh holds {pp['pop_share_pct']:.1f}% of the population on {pp['area_share_pct']:.2f}% of the land.")
print(f"2. Spearman r between density and household size: {r:.2f}.")
peak = nat.loc[nat['population'].idxmax()]
print(f"3. UN medium projection peaks at {peak['population']/1e6:.1f} M around {int(peak['year'])}.")